# Computer Vision Ödevi - YOLOv8 Nesne Tespiti (COCO)
Çalıştırma: Runtime > Change runtime type > **T4 GPU**, sonra Runtime > **Run all**. Hücreleri sırayla çalıştır.

In [ ]:
!pip -q install ultralytics gradio

## 1) Eğitim (YOLOv8n, COCO128 üzerinde fine-tune)

In [ ]:
import time
from ultralytics import YOLO
model = YOLO("yolov8n.pt")          # COCO ile ön eğitimli
t0 = time.time()
results = model.train(data="coco128.yaml", epochs=15, imgsz=640, batch=16,
                      optimizer="AdamW", lr0=0.001, project="runs", name="exp", plots=True)
train_time = time.time() - t0
print("Egitim suresi (sn):", round(train_time))

## 2) Değerlendirme (rapora yazılacak sayılar)

In [ ]:
best = YOLO(f"{results.save_dir}/weights/best.pt")
m = best.val(data="coco128.yaml")
print("Precision:", m.box.mp)
print("Recall   :", m.box.mr)
print("mAP@0.5  :", m.box.map50)
print("mAP@0.5:0.95:", m.box.map)
print("Tahmin suresi (ms/gorsel):", m.speed["inference"])
print("Egitim suresi (sn):", round(train_time))

## 3) Gradio uygulaması (fotoğraf yükle, sonucu gör)

In [ ]:
import gradio as gr
def predict(img):
    r = best(img, conf=0.25)[0]
    annotated = r.plot()[..., ::-1]     # BGR -> RGB
    names = [f"{best.names[int(c)]} ({float(p):.2f})" for c, p in zip(r.boxes.cls, r.boxes.conf)]
    return annotated, "\n".join(names) if names else "Nesne bulunamadi"
gr.Interface(predict, gr.Image(type="numpy"), [gr.Image(), gr.Textbox(label="Tespitler")],
             title="YOLOv8 Nesne Tespiti").launch(share=True, debug=True)